# Projection and Label Transfer

This notebook we will project the Resch data into HypoMap3D's embedding space and transfer labels.


### Imports

In [1]:
from pathlib import Path
import warnings
import os 

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scvi
from dotenv import dotenv_values
import torch
import scipy.sparse as sp

from hm3d_utils.clustering import project_query

warnings.simplefilter(action='ignore', category=FutureWarning)
sc.set_figure_params(frameon=False, figsize=(10,10), transparent=True)
scvi.settings.seed = 12345

use_gpu = torch.cuda.is_available()
print(f"GPU is available: {use_gpu}")

Seed set to 12345


GPU is available: True


### Config

In [ ]:
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

# Path to director of scVI model
SCVI_DIR  = DATADIR / "9_objects" / "scvi"

# Path to reference object
REF_FP = DATADIR / "9_objects" / "singlecell" / "hypomap3d_singlecell_control.h5ad" 

# PVH objects dir 
PVH_DIR = DATADIR.parent / "PVN_data_CellReports" 
OUTDIR = PVH_DIR / "projected"
#OUTDIR = DATADIR / "PVN_projected"
os.makedirs(OUTDIR, exist_ok=True)

### Let's load the reference (HypoMap3D)

In [3]:
ref_adata = sc.read_h5ad(REF_FP) 

In [4]:
ref_adata

AnnData object with n_obs × n_vars = 896959 × 56939
    obs: 'CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito', 'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link', 'Suspension', 'Technology', 'Author_Sex_harmonized', 'sex_per_sample_inferred', 'Strain', 'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'L1_ID', 'L2_ID', 'L3_ID', 'L4_ID', 'L5_ID', 'L1_descr', 'L2_descr', 'L3_descr', 'L4_descr', 'L5_descr', 'L1_abc_like', 'L2_abc_like', 'L3_abc_like', 'L4_abc_like', 'L5_abc_like'
    var: 'gene_ids', 'IsMouseTF', 'IsHumanTFOrtholog', 'IsSourceGene', 'IsTargetGene', 'IsHvgExcludedGene', 'IsAnnotationExcludedGene', 'IsPriorityGene', 'IsTopGene', 'IsHvg'
    obsm: 'X_embed', 'X_spatial_umap', 'X_umap'

### Let's get cell type label mappers

In [5]:
def build_id_mapping(n_df, id_pattern, target_pattern, levels=range(1, 6),):
    mapping = {}

    # --- 1. from wide dataframe ---
    for i in levels:
        id_col = id_pattern.format(i=i)
        target_col = target_pattern.format(i=i)

        if id_col in n_df.columns and target_col in n_df.columns:
            mapping.update(
                n_df[[id_col, target_col]]
                .dropna()
                .drop_duplicates()
                .set_index(id_col)[target_col]
                .to_dict()
            )

    return mapping
    
abclike_mapper = build_id_mapping(ref_adata.obs,
    id_pattern="L{i}_ID",
    target_pattern="L{i}_abc_like")

descr_mapper = build_id_mapping(ref_adata.obs,
    id_pattern="L{i}_ID",
    target_pattern="L{i}_descr") 


# Let's process all datasets in a loop

In [6]:

def build_map_matrix(leaves_to_level):

    leaves_to_level = np.asarray(leaves_to_level)
    level_names, inv = np.unique(leaves_to_level, return_inverse=True)
    rows = inv
    cols = np.arange(len(leaves_to_level))
    data = np.ones(len(leaves_to_level), dtype=np.uint8)
    M = sp.coo_matrix(
        (data, (rows, cols)),
        shape=(len(level_names), len(leaves_to_level)),
    ).tocsr()   

    return M, level_names

def propagate_confidences(adata, pathmat, P_path):

    # Build index maps

    level_maps = [] # 4 arrays indicating the cluster label of the path at Level i
    level_labels = {} # 4 keys, one for each level, mapping to arrays of unique cluster labels

    for lvl in range(4):
        labels = pathmat[:, lvl]
        level_labels[lvl] = np.unique(labels)
        level_maps.append(labels)

    # sum up 

    P_levels = {}

    for lvl in range(4):
        M, ids = build_map_matrix(level_maps[lvl])
        
        # (cells × leaves) @ (leaves × level_nodes)^T
        # Multiplying the cell × L5 probability matrix with M.T sums the
        # probabilities of all leaves belonging to the same parent cluster.
        P_lvl = P_path @ M.T
        
        P_levels[f"L{lvl+1}"] = P_lvl
        adata.uns[f"L{lvl+1}_ID"] = ids
        adata.uns[f"L{lvl+1}_descr"] = [descr_mapper[i] for i in ids]
        adata.uns[f"L{lvl+1}_abc_like"] = [abclike_mapper[i] for i in ids]
        adata.obsm[f"P_L{lvl+1}"] = P_lvl

    ids = pathmat[:, 4]
    adata.uns[f"L5_ID"] = ids
    adata.uns[f"L5_descr"] = [descr_mapper[i] for i in ids]
    adata.uns[f"L5_abc_like"] = [abclike_mapper[i] for i in ids]
    adata.obsm[f"P_L5"] = P_path

    for lvl in range(1, 6):
        P = adata.obsm[f"P_L{lvl}"]

        max_idx = np.asarray(P.argmax(axis=1)).ravel()
        max_prob = np.asarray(P.max(axis=1).toarray()).ravel()

        descr = np.asarray(adata.uns[f"L{lvl}_descr"])
        abc = np.asarray(adata.uns[f"L{lvl}_abc_like"])

        adata.obs[f"L{lvl}_descr"] = descr[max_idx]
        adata.obs[f"L{lvl}_abc_like"] = abc[max_idx]
        adata.obs[f"L{lvl}_max_p"] = max_prob

In [7]:
BATCH_COL_DICT = {
    "Seq_AllCell_Filtered": "sample",
    "MFISH_AllCell_Filtered": "Sample",
    "MFISH_Sim1": "Sample",
    "Seq_Sim1": "sample"
}

In [ ]:
for dset, BATCH_COL in BATCH_COL_DICT.items():

    print(f"Loading {dset}")
    query_adata = sc.read_h5ad(PVH_DIR / f"{dset}.h5ad") 
    query_all_genes = query_adata.copy()
    # ensure correct obs column name for batch key
    query_adata.obs = query_adata.obs.rename(columns={BATCH_COL: "Sample"})

    print("Setting up the model")
    query_adata = scvi.model.SCVI.prepare_query_anndata(query_adata, str(SCVI_DIR), inplace = False)

    # create the new query model instance.
    vae_query = scvi.model.SCVI.load_query_data(
        query_adata,
        str(SCVI_DIR)
    )
    # train
    print("Start training.")
    vae_query.train(max_epochs=20, plan_kwargs=dict(weight_decay=0.0))

    # get latent representation into query adata that has all genes
    query_all_genes.obsm["X_embed"] = vae_query.get_latent_representation()

    print("Transferring paths")
    pathmat, P_path = project_query(
    ref_adata,
    query_all_genes,
    path_keys=[f'L{i}_ID' for i in range(1,6)],
    k=50,
    use_rep='X_embed',
    use_gpu=use_gpu
    )
    query_all_genes.uns["pathmat"] = pathmat

    print("Propagating confidences")
    propagate_confidences(query_all_genes, pathmat, P_path)

    print("Saving object.")
    query_all_genes.write_h5ad(OUTDIR / f"{dset}_projected.h5ad")
